In [1]:
# ============================================================
# OLYMPIANS DATASET - NLP + K-MEANS CLUSTERING
# ============================================================

# ============================================================
# CELL 1 - INSTALL LIBRARIES
# ============================================================

# Run this cell if the libraries are not already installed.

# !pip install pandas numpy matplotlib seaborn scikit-learn scipy


# ============================================================
# CELL 2 - IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


# ============================================================
# CELL 3 - LOAD DATASET
# ============================================================

# Make sure olympians.csv is in the same folder as your
# Jupyter Notebook.

file_path = "olympians.csv"

df = pd.read_csv(file_path)

print("Dataset successfully loaded!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())


# ============================================================
# CELL 4 - BASIC INFORMATION
# ============================================================

print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nFirst 10 rows:")
display(df.head(10))


# ============================================================
# CELL 5 - REMOVE DUPLICATES
# ============================================================

print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))


# ============================================================
# CELL 6 - CLEAN COLUMN NAMES
# ============================================================

# Remove spaces and make column names easier to work with.

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Cleaned column names:")
print(df.columns.tolist())


# ============================================================
# CELL 7 - IDENTIFY TEXT AND NUMERICAL COLUMNS
# ============================================================

text_columns = df.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

print("Text columns:")
print(text_columns)

print("\nNumerical columns:")
print(numeric_columns)


# ============================================================
# CELL 8 - CLEAN TEXT COLUMNS
# ============================================================

for column in text_columns:
    df[column] = (
        df[column]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

print("Text columns cleaned.")


# ============================================================
# CELL 9 - CLEAN NUMERICAL COLUMNS
# ============================================================

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    df[column] = df[column].fillna(
        df[column].median()
    )

print("Numerical columns cleaned.")

display(df.head())


# ============================================================
# CELL 10 - CREATE NLP TEXT FIELD
# ============================================================

# We combine useful text columns into one column.
#
# This code automatically uses whichever of these columns
# exist in your dataset.

preferred_text_columns = [
    "Name",
    "Sex",
    "Team",
    "Sport",
    "Event",
    "Medal",
    "City",
    "Season",
    "Games"
]

available_text_columns = [
    column
    for column in preferred_text_columns
    if column in df.columns
]

print("Columns used for NLP:")
print(available_text_columns)

if len(available_text_columns) == 0:
    raise ValueError(
        "No suitable text columns were found in the dataset."
    )

df["combined_text"] = df[
    available_text_columns
].fillna("Unknown").astype(str).agg(
    " ".join,
    axis=1
)

print("\nExample NLP text:")
display(df[["combined_text"]].head())


# ============================================================
# CELL 11 - BASIC NLP CLEANING
# ============================================================

import re

def clean_text(text):
    text = str(text).lower()

    # Remove punctuation
    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["clean_text"] = df[
    "combined_text"
].apply(clean_text)

display(
    df[
        ["combined_text", "clean_text"]
    ].head()
)


# ============================================================
# CELL 12 - TF-IDF
# ============================================================

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=1000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

X_text = tfidf.fit_transform(
    df["clean_text"]
)

print("TF-IDF completed.")

print(
    "Number of documents:",
    X_text.shape[0]
)

print(
    "Number of NLP features:",
    X_text.shape[1]
)


# ============================================================
# CELL 13 - DISPLAY NLP FEATURES
# ============================================================

feature_names = tfidf.get_feature_names_out()

print("First 100 NLP features:")

for feature in feature_names[:100]:
    print(feature)


# ============================================================
# CELL 14 - IDENTIFY USEFUL NUMERICAL FEATURES
# ============================================================

# We normally do not want ID-like columns such as:
# ID, Athlete_ID, Year, etc. to dominate clustering.

excluded_numeric_columns = [
    "ID",
    "Id",
    "id",
    "Athlete_ID",
    "athlete_id"
]

clustering_numeric_columns = [
    column
    for column in numeric_columns
    if column not in excluded_numeric_columns
]

print("Numerical columns used for clustering:")
print(clustering_numeric_columns)


# ============================================================
# CELL 15 - STANDARDIZE NUMERICAL FEATURES
# ============================================================

if len(clustering_numeric_columns) > 0:

    numeric_data = df[
        clustering_numeric_columns
    ].copy()

    scaler = StandardScaler()

    X_numeric = scaler.fit_transform(
        numeric_data
    )

    print(
        "Numerical feature matrix:",
        X_numeric.shape
    )

else:

    X_numeric = None

    print(
        "No numerical features will be added."
    )


# ============================================================
# CELL 16 - COMBINE NLP + NUMERICAL FEATURES
# ============================================================

if X_numeric is not None:

    X = hstack([
        X_text,
        X_numeric
    ])

else:

    X = X_text


print(
    "Final feature matrix shape:",
    X.shape
)


# ============================================================
# CELL 17 - FIND OPTIMAL K USING ELBOW METHOD
# ============================================================

inertia = []

k_values = range(2, 11)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    inertia.append(
        model.inertia_
    )


# ============================================================
# CELL 18 - ELBOW PLOT
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    inertia,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method - Olympians Dataset"
)

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 19 - SILHOUETTE SCORE
# ============================================================

silhouette_scores = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(score)

    print(
        f"K = {k}, "
        f"Silhouette Score = {score:.4f}"
    )


# ============================================================
# CELL 20 - SILHOUETTE PLOT
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    silhouette_scores,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score - Olympians Dataset"
)

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 21 - AUTOMATICALLY SELECT K
# ============================================================

best_k = list(k_values)[
    np.argmax(silhouette_scores)
]

print(
    "Best K according to silhouette score:",
    best_k
)


# ============================================================
# CELL 22 - TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)

df["Cluster"] = cluster_labels

print("K-Means clustering completed.")


# ============================================================
# CELL 23 - CLUSTER COUNTS
# ============================================================

cluster_counts = (
    df["Cluster"]
    .value_counts()
    .sort_index()
)

print("Number of Olympians in each cluster:")

display(
    cluster_counts.to_frame(
        name="Number_of_Olympians"
    )
)


# ============================================================
# CELL 24 - PCA FOR VISUALIZATION
# ============================================================

# PCA requires a dense matrix.

X_dense = X.toarray()

print(
    "Dense matrix shape:",
    X_dense.shape
)


# ============================================================
# CELL 25 - PCA
# ============================================================

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_dense
)

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]

print(
    "Explained variance:",
    pca.explained_variance_ratio_
)

print(
    "Total explained variance:",
    pca.explained_variance_ratio_.sum()
)


# ============================================================
# CELL 26 - K-MEANS CLUSTER VISUALIZATION
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df,
    x="PCA1",
    y="PCA2",
    hue="Cluster",
    palette="tab10",
    s=60,
    alpha=0.7
)

plt.title(
    "Olympians - K-Means Clusters"
)

plt.xlabel(
    "Principal Component 1"
)

plt.ylabel(
    "Principal Component 2"
)

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 27 - ANALYSE EACH CLUSTER
# ============================================================

for cluster in sorted(
    df["Cluster"].unique()
):

    cluster_data = df[
        df["Cluster"] == cluster
    ]

    print("\n")
    print("=" * 70)
    print(
        f"CLUSTER {cluster}"
    )
    print("=" * 70)

    print(
        "Number of Olympians:",
        len(cluster_data)
    )

    # Sport
    if "Sport" in df.columns:

        print("\nTop Sports:")

        print(
            cluster_data[
                "Sport"
            ]
            .value_counts()
            .head(10)
        )

    # Event
    if "Event" in df.columns:

        print("\nTop Events:")

        print(
            cluster_data[
                "Event"
            ]
            .value_counts()
            .head(10)
        )

    # Team
    if "Team" in df.columns:

        print("\nTop Teams:")

        print(
            cluster_data[
                "Team"
            ]
            .value_counts()
            .head(10)
        )

    # Medal
    if "Medal" in df.columns:

        print("\nMedals:")

        print(
            cluster_data[
                "Medal"
            ]
            .value_counts()
        )


# ============================================================
# CELL 28 - TOP NLP WORDS FOR EACH CLUSTER
# ============================================================

terms = tfidf.get_feature_names_out()

for cluster in range(best_k):

    cluster_indices = np.where(
        df["Cluster"].values == cluster
    )[0]

    cluster_text = X_text[
        cluster_indices
    ]

    mean_tfidf = (
        cluster_text.mean(axis=0)
    )

    mean_tfidf = np.asarray(
        mean_tfidf
    ).flatten()

    top_indices = (
        mean_tfidf
        .argsort()[-20:][::-1]
    )

    print("\n")
    print("=" * 70)
    print(
        f"CLUSTER {cluster} - TOP NLP TERMS"
    )
    print("=" * 70)

    for index in top_indices:

        print(
            f"{terms[index]:25s}"
            f" {mean_tfidf[index]:.4f}"
        )


# ============================================================
# CELL 29 - SHOW SAMPLE OLYMPIANS FROM EACH CLUSTER
# ============================================================

columns_to_display = [
    "Name",
    "Sex",
    "Age",
    "Team",
    "Sport",
    "Event",
    "Medal",
    "Cluster"
]

available_display_columns = [
    column
    for column in columns_to_display
    if column in df.columns
]

for cluster in sorted(
    df["Cluster"].unique()
):

    print("\n")
    print("=" * 70)
    print(
        f"SAMPLE OLYMPIANS - CLUSTER {cluster}"
    )
    print("=" * 70)

    sample = df[
        df["Cluster"] == cluster
    ][available_display_columns].head(10)

    display(sample)


# ============================================================
# CELL 30 - CLUSTER DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="Cluster"
)

plt.title(
    "Number of Olympians per Cluster"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Olympians"
)

plt.show()


# ============================================================
# CELL 31 - SPORTS BY CLUSTER
# ============================================================

if "Sport" in df.columns:

    sport_cluster = pd.crosstab(
        df["Sport"],
        df["Cluster"]
    )

    print(
        "Sports distribution across clusters:"
    )

    display(
        sport_cluster.head(30)
    )


# ============================================================
# CELL 32 - MEDALS BY CLUSTER
# ============================================================

if "Medal" in df.columns:

    medal_cluster = pd.crosstab(
        df["Cluster"],
        df["Medal"]
    )

    print(
        "Medal distribution across clusters:"
    )

    display(
        medal_cluster
    )


# ============================================================
# CELL 33 - AGE ANALYSIS
# ============================================================

if "Age" in df.columns:

    plt.figure(figsize=(10, 6))

    sns.boxplot(
        data=df,
        x="Cluster",
        y="Age"
    )

    plt.title(
        "Age Distribution by Cluster"
    )

    plt.xlabel(
        "Cluster"
    )

    plt.ylabel(
        "Age"
    )

    plt.show()


# ============================================================
# CELL 34 - CLUSTER SUMMARY
# ============================================================

summary = []

for cluster in sorted(
    df["Cluster"].unique()
):

    cluster_data = df[
        df["Cluster"] == cluster
    ]

    row = {
        "Cluster": cluster,
        "Number_of_Olympians": len(cluster_data)
    }

    if "Age" in df.columns:

        row["Average_Age"] = (
            cluster_data["Age"]
            .mean()
        )

    if "Height" in df.columns:

        row["Average_Height"] = (
            cluster_data["Height"]
            .mean()
        )

    if "Weight" in df.columns:

        row["Average_Weight"] = (
            cluster_data["Weight"]
            .mean()
        )

    if "Sport" in df.columns:

        row["Most_Common_Sport"] = (
            cluster_data["Sport"]
            .mode()
            .iloc[0]
            if not cluster_data["Sport"].mode().empty
            else "Unknown"
        )

    if "Medal" in df.columns:

        row["Most_Common_Medal"] = (
            cluster_data["Medal"]
            .mode()
            .iloc[0]
            if not cluster_data["Medal"].mode().empty
            else "Unknown"
        )

    summary.append(row)


cluster_summary = pd.DataFrame(
    summary
)

display(
    cluster_summary
)


# ============================================================
# CELL 35 - SAVE RESULTS
# ============================================================

output_file = "olympians_kmeans_nlp_results.csv"

df.to_csv(
    output_file,
    index=False
)

print(
    f"Results saved to: {output_file}"
)


# ============================================================
# CELL 36 - SAVE CLUSTER SUMMARY
# ============================================================

summary_file = "olympians_cluster_summary.csv"

cluster_summary.to_csv(
    summary_file,
    index=False
)

print(
    f"Cluster summary saved to: {summary_file}"
)


# ============================================================
# CELL 37 - FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "Original dataset rows:",
    len(df)
)

print(
    "Number of clusters:",
    best_k
)

print(
    "Silhouette score:",
    max(silhouette_scores)
)

print(
    "\nCluster sizes:"
)

display(
    cluster_counts.to_frame(
        name="Olympians"
    )
)

print(
    "\nCluster summary:"
)

display(
    cluster_summary
)

print(
    "\nFiles created:"
)

print(
    "1. olympians_kmeans_nlp_results.csv"
)

print(
    "2. olympians_cluster_summary.csv"
)

Dataset successfully loaded!
Number of rows: 11538
Number of columns: 12


,id,name,nationality,sex,date_of_birth,height,weight,sport,gold,silver,bronze,info
0,736041664,A Jesus Garcia,ESP,male,1969-10-17,1.72,64.0,athletics,0,0,0,NaN
1,532037425,A Lam Shin,KOR,female,1986-09-23,1.68,56.0,fencing,0,0,0,NaN
2,435962603,Aaron Brown,CAN,male,1992-05-27,1.98,79.0,athletics,0,0,1,NaN
3,521041435,Aaron Cook,MDA,male,1991-01-02,1.83,80.0,taekwondo,0,0,0,NaN
4,33922579,Aaron Gate,NZL,male,1990-11-26,1.81,71.0,cycling,0,0,0,NaN


Dataset shape:
(11538, 12)

Column names:
['id', 'name', 'nationality', 'sex', 'date_of_birth', 'height', 'weight', 'sport', 'gold', 'silver', 'bronze', 'info']

Data types:
id                 int64
name                 str
nationality          str
sex                  str
date_of_birth        str
height           float64
weight           float64
sport                str
gold               int64
silver             int64
bronze             int64
info                 str
dtype: object

Missing values:


id                   0
name                 0
nationality          0
sex                  0
date_of_birth        0
height             330
weight             659
sport                0
gold                 0
silver               0
bronze               0
info             11407
dtype: int64


First 10 rows:


...

Rows before removing duplicates: 11538
Rows after removing duplicates: 11538
Cleaned column names:
['id', 'name', 'nationality', 'sex', 'date_of_birth', 'height', 'weight', 'sport', 'gold', 'silver', 'bronze', 'info']
Text columns:
['name', 'nationality', 'sex', 'date_of_birth', 'sport', 'info']

Numerical columns:
['id', 'height', 'weight', 'gold', 'silver', 'bronze']
Text columns cleaned.
Numerical columns cleaned.


/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_94233/208006614.py:102: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(


,id,name,nationality,sex,date_of_birth,height,weight,sport,gold,silver,bronze,info
0,736041664,A Jesus Garcia,ESP,male,1969-10-17,1.72,64.0,athletics,0,0,0,Unknown
1,532037425,A Lam Shin,KOR,female,1986-09-23,1.68,56.0,fencing,0,0,0,Unknown
2,435962603,Aaron Brown,CAN,male,1992-05-27,1.98,79.0,athletics,0,0,1,Unknown
3,521041435,Aaron Cook,MDA,male,1991-01-02,1.83,80.0,taekwondo,0,0,0,Unknown
4,33922579,Aaron Gate,NZL,male,1990-11-26,1.81,71.0,cycling,0,0,0,Unknown


Columns used for NLP:
[]


ValueError: No suitable text columns were found in the dataset.